# Capstone Project - Feature Selection and Engineering

This notebook covers feature selection and engineering for my diabetes risk prediction capstone.

The main goals of this notebook are to:

1. Reload and prepare the dataset in a clean and reproducible way
2. Apply feature engineering based on the findings from my EDA
3. Define Feature Sets A, B, and C for the modelling notebooks

This notebook is kept separate from the EDA notebook so that the modelling pipeline can run independently from start to finish.

I explain my observations and reasoning for feature selection and engineering here. My separate modelling notebooks (`CDM Feature A Modelling.ipynb`, `CDM Feature B Modeling.ipynb`, and `CDM Feature C Modelling.ipynb`) focus on model results rather than repeating these feature engineering steps.


## 1.0 Loading the Data


In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier,
    VotingClassifier,
    StackingClassifier
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    roc_curve
)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print("All libraries loaded successfully.")

All libraries loaded successfully.


In [2]:
repo = fetch_ucirepo(id=891)

X_raw = repo.data.features
y_raw = repo.data.targets

df = pd.concat([X_raw, y_raw], axis=1)

print("Dataset loaded successfully.")
print("Original shape:", df.shape)

Dataset loaded successfully.
Original shape: (253680, 22)


In [3]:
# Rename columns into easier format
df = df.rename(columns={
    'HighBP': 'high_bp',
    'HighChol': 'high_chol',
    'CholCheck': 'chol_check',
    'BMI': 'bmi',
    'Smoker': 'smoker',
    'Stroke': 'stroke',
    'HeartDiseaseorAttack': 'heart_disease',
    'PhysActivity': 'phys_activity',
    'Fruits': 'fruits',
    'Veggies': 'veggies',
    'HvyAlcoholConsump': 'heavy_alcohol',
    'AnyHealthcare': 'any_healthcare',
    'NoDocbcCost': 'no_doc_cost',
    'GenHlth': 'general_health',
    'MentHlth': 'mental_health_days',
    'PhysHlth': 'physical_health_days',
    'DiffWalk': 'difficulty_walking',
    'Sex': 'sex',
    'Age': 'age',
    'Education': 'education',
    'Income': 'income',
    'Diabetes_binary': 'diabetes'
})

df.head()

,high_bp,high_chol,chol_check,bmi,smoker,stroke,heart_disease,phys_activity,fruits,veggies,...,no_doc_cost,general_health,mental_health_days,physical_health_days,difficulty_walking,sex,age,education,income,diabetes
0,1,1,1,40,1,0,0,0,0,1,...,0,5,18,15,1,0,9,4,3,0
1,0,0,0,25,1,0,0,1,0,0,...,1,3,0,0,0,0,7,6,1,0
2,1,1,1,28,0,0,0,0,1,0,...,1,5,30,30,1,0,9,4,8,0
3,1,0,1,27,0,0,0,1,1,1,...,0,2,0,0,0,0,11,3,6,0
4,1,1,1,24,0,0,0,1,1,1,...,0,2,3,0,0,0,11,5,4,0


In [4]:
# Check duplicate rows
duplicate_count = df.duplicated().sum()
duplicate_percent = (duplicate_count / len(df)) * 100

print(f"Duplicate rows: {duplicate_count} ({duplicate_percent:.2f}% of the dataset)")

Duplicate rows: 24206 (9.54% of the dataset)


In [5]:
# Drop duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

print("Shape after dropping duplicates:", df.shape)

Shape after dropping duplicates: (229474, 22)


### Observation

I repeated the same cleaning step from my EDA notebook by removing duplicate rows. I wanted this notebook to be self-contained and reproducible, rather than depending on my EDA notebook.

This also means I am using the same cleaned version of the dataset when moving from exploration into modelling.


## 2.0 Defining Features and Target


In [6]:
X = df.drop('diabetes', axis=1)
y = df['diabetes']

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (229474, 21)
Target shape: (229474,)


In [7]:
target_summary = y.value_counts().to_frame(name='count')
target_summary['percentage'] = (y.value_counts(normalize=True) * 100).round(2)

target_summary

,count,percentage
diabetes,,
0,194377,84.71
1,35097,15.29


## 3.0 Feature Engineering Plan


The feature engineering in this notebook is based directly on the patterns that stood out in the EDA notebook.

The EDA suggested that the strongest relationships with my target variable were coming from the following features:

- BMI
- high blood pressure
- high cholesterol
- general health
- age
- difficulty walking
- physical activity
- income and education

Because of that, I want to test three different feature sets:

### Feature Set A
The original cleaned features only (21 predictors)

### Feature Set B
The original cleaned features plus engineered features (27 predictors, with raw BMI replaced by BMI category dummies)

### Feature Set C
A reduced feature set using the strongest predictors from the EDA (14 predictors)

This will allow me to determine which set of features has the strongest predictive effect on my target variable, and whether my engineered features improve model performance.


## 4.0 Creating Engineered Features


In [8]:
# Create BMI categories using standard clinical cut-offs
bmi_bins = [0, 18.5, 25, 30, 35, 40, float('inf')]
bmi_labels = [
    'Underweight',
    'Normal weight',
    'Overweight',
    'Obese Class I',
    'Obese Class II',
    'Obese Class III'
]

df['bmi_category'] = pd.cut(
    df['bmi'],
    bins=bmi_bins,
    labels=bmi_labels,
    right=False
)

df[['bmi', 'bmi_category']].sort_values('bmi').head(10)

,bmi,bmi_category
216828,12,Underweight
126181,12,Underweight
192325,12,Underweight
162771,12,Underweight
92077,12,Underweight
47643,12,Underweight
37416,13,Underweight
38933,13,Underweight
190473,13,Underweight
116933,13,Underweight


In [9]:
risk_features = [
    'high_bp',
    'high_chol',
    'heart_disease',
    'difficulty_walking',
    'stroke'
]

df['risk_factor_count'] = df[risk_features].sum(axis=1)

df[['risk_factor_count']].describe()

,risk_factor_count
count,229474.000000
mean,1.229887
std,1.146698
min,0.000000
25%,0.000000
50%,1.000000
75%,2.000000
max,5.000000


In [10]:
df['overall_health_burden'] = (
    df['general_health'] +
    df['mental_health_days'] +
    df['physical_health_days']
)

df[['general_health', 'mental_health_days', 'physical_health_days', 'overall_health_burden']].head()

,general_health,mental_health_days,physical_health_days,overall_health_burden
0,5,18,15,38
1,3,0,0,3
2,5,30,30,65
3,2,0,0,2
4,2,3,0,5


### Observation

I created these new features based on the findings from the EDA:

- `bmi_category` gives a grouped version of BMI using standard clinical ranges.
- `risk_factor_count` combines the features that showed the strongest relationship with the target variable into one feature.
- `overall_health_burden` combines general health, mental health days, and physical health days into one overall health burden score.

I do not want to assume these features will automatically improve the models, so I will use three different feature sets in my modelling notebooks:

**Feature Set A** has all 21 original variables.

**Feature Set B** adds my engineered features (27 predictors total, after replacing raw BMI with BMI category dummies) to see whether they improve model performance.

**Feature Set C** keeps only the 14 strongest EDA-selected predictors.

The different feature sets are discussed below.


## 5.0 Encoding the New Categorical Feature


In [11]:
# One-hot encode the BMI category feature
df_model = pd.get_dummies(df, columns=['bmi_category'], drop_first=True)

df_model.head()

,high_bp,high_chol,chol_check,bmi,smoker,stroke,heart_disease,phys_activity,fruits,veggies,...,education,income,diabetes,risk_factor_count,overall_health_burden,bmi_category_Normal weight,bmi_category_Overweight,bmi_category_Obese Class I,bmi_category_Obese Class II,bmi_category_Obese Class III
0,1,1,1,40,1,0,0,0,0,1,...,4,3,0,3,38,False,False,False,False,True
1,0,0,0,25,1,0,0,1,0,0,...,6,1,0,0,3,False,True,False,False,False
2,1,1,1,28,0,0,0,0,1,0,...,4,8,0,3,65,False,True,False,False,False
3,1,0,1,27,0,0,0,1,1,1,...,3,6,0,1,2,False,True,False,False,False
4,1,1,1,24,0,0,0,1,1,1,...,5,4,0,2,5,True,False,False,False,False


### Observation

I used one-hot encoding for the BMI category feature because the models need numeric inputs rather than text categories.

I dropped the first category to avoid unnecessary duplication between the dummy variables.


## 6.0 Creating Feature Sets


In [12]:
# Feature Set A: original cleaned features only
feature_set_a = [
    'high_bp', 'high_chol', 'chol_check', 'bmi', 'smoker', 'stroke',
    'heart_disease', 'phys_activity', 'fruits', 'veggies',
    'heavy_alcohol', 'any_healthcare', 'no_doc_cost',
    'general_health', 'mental_health_days', 'physical_health_days',
    'difficulty_walking', 'sex', 'age', 'education', 'income'
]

In [13]:
# Feature Set B: original features plus engineered features
# Raw BMI removed because BMI is now represented through clinical BMI categories
feature_set_b = [
    'high_bp', 'high_chol', 'chol_check', 'smoker', 'stroke',
    'heart_disease', 'phys_activity', 'fruits', 'veggies',
    'heavy_alcohol', 'any_healthcare', 'no_doc_cost',
    'general_health', 'mental_health_days', 'physical_health_days',
    'difficulty_walking', 'sex', 'age', 'education', 'income',
    'risk_factor_count',
    'overall_health_burden',
    'bmi_category_Normal weight',
    'bmi_category_Overweight',
    'bmi_category_Obese Class I',
    'bmi_category_Obese Class II',
    'bmi_category_Obese Class III'
]

### Observation

For Feature Set B, I chose to remove the raw `bmi` variable once I added the BMI category dummy variables.

I did this because both the raw BMI value and the BMI categories represent the same underlying information. Keeping both could create redundancy and make the results less generalisable. By using the clinical BMI categories instead, I can test whether the grouped version of BMI is more useful than the raw numeric value.


In [14]:
# Feature Set C: reduced feature set based on EDA
feature_set_c = [
    'bmi',
    'high_bp',
    'high_chol',
    'chol_check',
    'heart_disease',
    'stroke',
    'phys_activity',
    'general_health',
    'mental_health_days',
    'physical_health_days',
    'difficulty_walking',
    'age',
    'education',
    'income',
]

## 7.0 Defining Target Variable and Feature Sets


In [15]:
y = df_model['diabetes']

X_a = df_model[feature_set_a]
X_b = df_model[feature_set_b]
X_c = df_model[feature_set_c]

print("Feature Set A shape:", X_a.shape)
print("Feature Set B shape:", X_b.shape)
print("Feature Set C shape:", X_c.shape)

Feature Set A shape: (229474, 21)
Feature Set B shape: (229474, 27)
Feature Set C shape: (229474, 14)


## 8.0 Reproducible Pipeline Example

The modelling notebooks scale and train models in separate steps. For deployment and reproducibility, the same logic can be wrapped in a single `Pipeline` so preprocessing and prediction stay aligned.

Note: in the modelling notebooks I avoided `probability=True` on RBF SVM because it can be very slow on this dataset. For ROC AUC I used `decision_function` instead. The example below keeps that same practical setting.


In [16]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

example_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(
        kernel='rbf',
        class_weight='balanced',
        random_state=42
    ))
])

print('Pipeline steps:', [name for name, _ in example_pipeline.steps])
print('Use the same pattern to save/load for batch scoring in production.')

Pipeline steps: ['scaler', 'model']
Use the same pattern to save/load for batch scoring in production.


### Conclusion on Feature Sets

After my EDA I decided to split my features into three different sets as follows:

- **Feature Set A** has all 21 original features.
- **Feature Set B** tests whether the engineered features add any real predictive value to the original features (27 predictors, with raw BMI replaced by clinical BMI categories).
- **Feature Set C** tests whether the 14 features with the strongest relationships with the target variable can still support strong model performance.

Overall, this will allow me to see how different feature sets perform and understand which features have the most influence on my target variable. The modelling notebooks then compare nine models on each set, prioritising recall and false negatives for screening.
